# Compare the two trace generators

Run the cells from top to bottom using a Python environment containing **NumPy, pandas, and Matplotlib** (your MLOps environment).

- `generate_random_trace.generate()` creates the static CSV directly.
- `Workload.requests_between()` returns generated request descriptions for a simulated time interval. Running `workload.py` by itself does not generate a file.

Here we call both implementations directly. The workload clock advances in fixed steps, so there is **no real-time waiting and no FastAPI server**. This examines the generator itself rather than the HTTP recording path.

Both samples cover the same simulated duration. Their traffic rates, token distributions, SLA values, and batch behavior differ by design, so equal duration does **not** mean equal row counts or equal total work. The notebook uses temporary CSVs and leaves your versioned traces, manifests, latest pointers, and lineage untouched.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print  # Also lets the code run outside a notebook.

# Find the repository whether the notebook starts here or at the repository root.
REPO_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "mlops" / "trace_sim" / "workload.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the Sage-Serve repository.")

mlops_path = str(REPO_ROOT / "mlops")
if mlops_path not in sys.path:
    sys.path.insert(0, mlops_path)

from trace_sim.generate_random_trace import generate
from trace_sim.workload import Workload, COLUMNS

print("Repository:", REPO_ROOT)
print("Python interpreter:", sys.executable)

## 1. Choose a small experiment

The defaults reproduce the static generator's **5,000 iterations with seed 1**, alongside 5,000 simulated seconds from `Workload`.

The static generator assigns timestamp `i` to iteration `i`, so using the duration as its iteration count aligns the time spans. A batch iteration writes 128 rows, so this is not a 5,000-row limit.

`STEP_SECONDS=12` asks the workload for arrivals twelve simulated seconds at a time, similar to the streaming runner's nominal 0.2-second tick at 60× speedup. Fixed steps make this notebook reproducible for fixed settings and library versions. The same seed does not make the two generators produce matching requests.

To see a full daily pattern, try `DURATION_SECONDS = 24 * 3600`; this also produces a substantially larger static CSV. The short default does not demonstrate weekend behavior.

In [ ]:
SEED = 1
DURATION_SECONDS = 5000
STEP_SECONDS = 12
RATE_SCALE = 0.002

assert isinstance(DURATION_SECONDS, int) and DURATION_SECONDS > 0
assert STEP_SECONDS > 0 and RATE_SCALE > 0
print(f"Comparing {DURATION_SECONDS / 3600:.2f} simulated hours.")

## 2. Generate the static sample

`generate()` writes a CSV at the path we supply. We read it into a DataFrame (a table in memory) before the temporary directory is removed. Reading `regions` as text preserves leading zeros such as `012`.

In [ ]:
with TemporaryDirectory(prefix="sageserve-static-") as temporary_folder:
    csv_path = Path(temporary_folder) / "static_trace.csv"
    static_rows_written = generate(DURATION_SECONDS, SEED, csv_path)
    static_df = pd.read_csv(csv_path, dtype={"regions": "string"})

print(f"Static generator: {static_rows_written:,} CSV rows.")

## 3. Generate the workload sample

For every time interval, `requests_between(start, end)` returns `(region, model, row)` tuples. The `row` dictionary already contains the fifteen trace fields, including the region preference and model.

We collect those rows, put them in chronological order for inspection, and round-trip them through a temporary CSV. Batch drops may occur exactly at an interval's end, including the experiment's final boundary. Burst and batch messages are collected separately below.

In [ ]:
workload = Workload(rate_scale=RATE_SCALE, seed=SEED)
rows = []
workload_events = []

for start in range(0, DURATION_SECONDS, STEP_SECONDS):
    end = min(start + STEP_SECONDS, DURATION_SECONDS)
    for region, model, row in workload.requests_between(start, end):
        rows.append(row)
    for message in workload.events:
        workload_events.append({"interval_end_seconds": end, "event": message})
    workload.events.clear()

workload_df = pd.DataFrame(rows, columns=COLUMNS)
if workload_df.empty:
    raise ValueError("No workload arrivals generated. Increase duration or rate scale.")
workload_df = workload_df.sort_values("arrival_timestamp", kind="stable").reset_index(drop=True)

with TemporaryDirectory(prefix="sageserve-workload-") as temporary_folder:
    csv_path = Path(temporary_folder) / "workload_trace.csv"
    workload_df.to_csv(csv_path, index=False)
    workload_df = pd.read_csv(csv_path, dtype={"regions": "string"})

# Basic checks that both samples really use the common trace schema.
datasets = {"Static generator": static_df, "Workload generator": workload_df}
for name, df in datasets.items():
    assert list(df.columns) == COLUMNS, name
    assert df.notna().all().all(), name
    assert df["regions"].isin(["012", "021", "102", "120", "201", "210"]).all(), name
    assert df["arrival_timestamp"].between(0, DURATION_SECONDS).all(), name
assert len(static_df) == static_rows_written

print(f"Workload generator: {len(workload_df):,} CSV rows.")

## 4. Inspect rows side by side

Each half shows the first eight records from one generator. These rows are independent samples, not matched requests. To inspect all fifteen fields for either sample, evaluate `static_df.head()` or `workload_df.head()` in a new cell.

In [ ]:
columns_to_compare = [
    "request_id", "batch_id", "arrival_timestamp", "model_type", "regions",
    "sla", "workload_type", "prompt_size", "token_size",
]
display(pd.concat(
    {name: df[columns_to_compare].head(8).reset_index(drop=True)
     for name, df in datasets.items()},
    axis=1,
))

## 5. Compare counts and sizes

A CSV row is a record. A unique request ID is a distinct identifier. They differ for the static generator because its 128-row batches reuse an ID. The workload generator assigns each row a fresh ID. We retain every row, just as the forecasting aggregation does; we do not deduplicate or multiply by `batch_size`.

In [ ]:
summary = pd.DataFrame({
    name: {
        "CSV rows": len(df),
        "Unique request IDs": df["request_id"].nunique(),
        "Rows beyond unique IDs": len(df) - df["request_id"].nunique(),
        "Unique batches": df.loc[df["batch_id"] >= 0, "batch_id"].nunique(),
        "Rows belonging to batches": int((df["batch_id"] >= 0).sum()),
        "First arrival (seconds)": df["arrival_timestamp"].min(),
        "Last arrival (seconds)": df["arrival_timestamp"].max(),
        "Mean prompt tokens per row": df["prompt_size"].mean(),
        "Mean output tokens per row": df["token_size"].mean(),
        "Total prompt tokens": df["prompt_size"].sum(),
    }
    for name, df in datasets.items()
})
display(summary.round(2))

## 6. Compare arrivals and prompt demand over time

The upper plots count **CSV records per one-minute bucket**. The lower plots sum **prompt tokens per one-minute bucket**, the signal used by the forecaster. Empty minutes are included as zeros. A final bucket may contain less than a full minute; a batch at the final boundary is retained.

Each column has its own vertical scale so the pattern remains visible. Read those scales when comparing magnitude. The generators' rates have not been calibrated to match.

In [ ]:
minute_index = pd.RangeIndex(DURATION_SECONDS // 60 + 1, name="minute")
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)

for column, (name, df) in enumerate(datasets.items()):
    minute = (df["arrival_timestamp"] // 60).astype(int)
    records = df.groupby(minute).size().reindex(minute_index, fill_value=0)
    prompts = df.groupby(minute)["prompt_size"].sum().reindex(minute_index, fill_value=0)
    axes[0, column].plot(records.index, records.values, linewidth=1)
    axes[1, column].plot(prompts.index, prompts.values, linewidth=1, color="tab:orange")
    axes[0, column].set_title(name)
    axes[0, column].set_ylabel("CSV rows / minute bucket")
    axes[1, column].set_ylabel("Prompt tokens / minute bucket")
    axes[1, column].set_xlabel("Simulated minute since Monday 00:00")
    for ax in axes[:, column]:
        ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 7. Compare token sizes and model/region shares

The cumulative curves answer: **what fraction of records have a token size at or below this value?** A curve farther to the right means larger sizes. The horizontal axis uses a logarithmic scale to show both small and large requests.

The bar charts compare percentages of CSV rows, so different total row counts do not obscure model and first-priority-region shares.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, field, title in zip(axes, ["prompt_size", "token_size"], ["Input prompt size", "Output size"]):
    for name, df in datasets.items():
        values = np.sort(df[field].to_numpy())
        fraction = np.arange(1, len(values) + 1) / len(values)
        ax.plot(values, fraction, label=name)
    ax.set_xscale("log")
    ax.set_xlabel("Tokens per CSV row (log scale)")
    ax.set_ylabel("Fraction of rows at or below size")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
plt.show()

model_shares = pd.DataFrame({
    name: df["model_type"].value_counts(normalize=True).reindex(list("ABCD"), fill_value=0) * 100
    for name, df in datasets.items()
})
region_shares = pd.DataFrame({
    name: df["regions"].str[0].value_counts(normalize=True).reindex(list("012"), fill_value=0) * 100
    for name, df in datasets.items()
}).rename(index={"0": "West", "1": "Central", "2": "East"})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
model_shares.plot.bar(ax=axes[0], rot=0, ylabel="Share of CSV rows (%)", title="Model shares")
region_shares.plot.bar(ax=axes[1], rot=0, ylabel="Share of CSV rows (%)", title="First-priority region shares")
fig.tight_layout()
plt.show()

## 8. Check SLA values and workload events

The generators use different SLA settings: static values include `10`, `3600`, `86400`, and `-1`; workload values include `1`, `60`, and `86400`. These are seconds; `-1` is the static generator's development-request sentinel. We display the actual values instead of assigning both sources identical tier labels.

`prod`/`dev` and SLA tier are separate fields. A non-interactive request can belong to production.

In [ ]:
sla_shares = pd.DataFrame({
    name: df["sla"].value_counts(normalize=True) * 100
    for name, df in datasets.items()
}).fillna(0).sort_index()
sla_shares.index.name = "SLA value (seconds; -1 is a sentinel)"
display(sla_shares.round(2).rename_axis(columns="CSV row share (%)"))

if workload_events:
    display(pd.DataFrame(workload_events).head(20))
else:
    print("No bursts or batch drops occurred in this short workload sample; that is a valid random outcome.")

You have now run both actual generator implementations and inspected their outputs. The differences in row counts, IDs, time patterns, SLA values, and token sizes come from their different rules.

The temporary CSVs have been removed, while `static_df` and `workload_df` remain available in memory for further exploration. Re-run from the top after changing settings. This notebook does not call `ingest()` or `finalize()`, so it does not create pipeline versions or move `latest.json`.

## 9. Generate a full week of CSV traces from BOTH generators

The first comparison covers about 83 minutes on Monday. To compare the two generators across Monday–Sunday, we now create **two actual CSV traces covering the same seven simulated days**:

1. Call `generate(604800, seed, path)` for the static trace. The generator assigns timestamp `i` to iteration `i`, so 604,800 iterations cover seven days. Batch iterations expand into 128 rows.
2. Call a fresh `Workload.requests_between()` repeatedly across that same week and write every returned row into the workload CSV.
3. In the following section, read **both CSVs** in chunks and calculate all plotted counts from those saved rows.

Timestamp 0 is labeled Monday 00:00 in both plots. This is an alignment convention for the static trace; the static generator itself has no weekday/weekend logic. `Workload` explicitly uses that calendar.

**Size and storage:** the static trace contains roughly eight million rows (hundreds of MB). CSVs are stored in a temporary directory, outside versioned `data/`, and their paths are printed. They remain available while `weekly_csv_storage` exists in the kernel, and are removed when that object is cleaned up. We read in chunks to keep memory usage modest. No manifests, latest pointers, or lineage are changed. There is no real-time waiting or FastAPI call.

In [ ]:
import csv

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
WEEK_SECONDS = 7 * 24 * 3600
WEEK_SEED = SEED
WEEK_RATE_SCALE = RATE_SCALE
WEEK_STEP_SECONDS = STEP_SECONDS
CSV_CHUNK_SIZE = 100_000

assert isinstance(WEEK_STEP_SECONDS, int) and WEEK_STEP_SECONDS > 0
assert WEEK_RATE_SCALE > 0

weekly_csv_storage = TemporaryDirectory(prefix="sageserve-weekly-traces-")
weekly_csv_dir = Path(weekly_csv_storage.name)
weekly_trace_paths = {
    "Static generator": weekly_csv_dir / "static_week.csv",
    "Workload generator": weekly_csv_dir / "workload_week.csv",
}

print("Generating the static CSV for seven simulated days...", flush=True)
weekly_static_rows = generate(WEEK_SECONDS, WEEK_SEED, weekly_trace_paths["Static generator"])
print(f"Static CSV: {weekly_static_rows:,} rows", flush=True)

weekly_workload = Workload(rate_scale=WEEK_RATE_SCALE, seed=WEEK_SEED)
weekly_workload_rows = 0
last_reported_day = -1
print("Generating the workload CSV...", flush=True)
with weekly_trace_paths["Workload generator"].open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=COLUMNS, lineterminator="\n")
    writer.writeheader()
    for start in range(0, WEEK_SECONDS, WEEK_STEP_SECONDS):
        end = min(start + WEEK_STEP_SECONDS, WEEK_SECONDS)
        for region, model, row in weekly_workload.requests_between(start, end):
            writer.writerow(row)
            weekly_workload_rows += 1
        weekly_workload.events.clear()
        completed_day = end // 86400 - 1
        if completed_day > last_reported_day:
            print(f"  Finished {DAY_NAMES[completed_day]}", flush=True)
            last_reported_day = completed_day

for name, path in weekly_trace_paths.items():
    print(f"{name}: {path} ({path.stat().st_size / 1024**2:.1f} MiB)")

## 10. Read the saved traces and count arrivals

A one-minute bucket groups all rows whose timestamps fall within that minute. We count CSV rows and sum prompt tokens in each bucket. Empty minutes stay present as zeros. Hourly counts add 60 minute buckets; daily counts add 24 hourly buckets.

This is the **same calculation for both CSVs**. Every figure below uses those read-back counts, not the generator's configured arrival rates. A workload batch may land on the exact next-Monday boundary; such rows remain in its CSV but are excluded from this seven-day analysis, which covers `0 <= timestamp < 604800`.

The comparison unit is **CSV rows**, because the static generator repeats request IDs inside batches while the workload generator gives each row a new ID. Row counts measure the records actually generated and retain the batch expansion used by the forecasting input. They are not equivalent to unique static request IDs. The generators' overall traffic volumes are not matched.

In [ ]:
weekly_minutes_by_generator = {}
weekly_hours_by_generator = {}
weekly_file_summary = []

for name, path in weekly_trace_paths.items():
    assert list(pd.read_csv(path, nrows=0).columns) == COLUMNS, name
    minute_rows = np.zeros(7 * 24 * 60, dtype=np.int64)
    minute_prompt_tokens = np.zeros_like(minute_rows)
    total_read = 0
    excluded_rows = 0

    for chunk in pd.read_csv(
        path, usecols=["arrival_timestamp", "prompt_size"], chunksize=CSV_CHUNK_SIZE
    ):
        total_read += len(chunk)
        assert chunk.notna().all().all(), name
        in_week = chunk["arrival_timestamp"].between(0, WEEK_SECONDS, inclusive="left")
        excluded_rows += int((~in_week).sum())
        valid = chunk.loc[in_week]
        minute = (valid["arrival_timestamp"] // 60).astype(int)
        counts = valid.groupby(minute).size()
        tokens = valid.groupby(minute)["prompt_size"].sum()
        minute_rows[counts.index.to_numpy()] += counts.to_numpy()
        minute_prompt_tokens[tokens.index.to_numpy()] += tokens.to_numpy(dtype=np.int64)

    minutes = pd.DataFrame({"CSV rows": minute_rows, "Prompt tokens": minute_prompt_tokens})
    minutes.index.name = "minute"
    hours = minutes.groupby(minutes.index // 60).sum()
    hours.index.name = "hour"
    assert len(minutes) == 10080 and len(hours) == 168
    assert int(minutes["CSV rows"].sum()) + excluded_rows == total_read
    expected_written = weekly_static_rows if name == "Static generator" else weekly_workload_rows
    assert total_read == expected_written, (name, total_read, expected_written)
    weekly_minutes_by_generator[name] = minutes
    weekly_hours_by_generator[name] = hours
    weekly_file_summary.append({
        "Generator": name, "CSV rows read": total_read,
        "Rows inside the week": int(minutes["CSV rows"].sum()),
        "Rows outside the week": excluded_rows,
    })

display(pd.DataFrame(weekly_file_summary).set_index("Generator"))
weekly_daily_rows = pd.DataFrame({
    name: hours["CSV rows"].groupby(hours.index // 24).sum().to_numpy()
    for name, hours in weekly_hours_by_generator.items()
}, index=pd.Index(DAY_NAMES, name="Day"))
print("Daily CSV row counts, read from the generated traces:")
display(weekly_daily_rows)

## 11. Compare the complete weekly time series side by side

Each time-series point is the mean CSV row arrival rate during one complete simulated hour (`rows in that hour / 3600`). Shading marks Saturday and Sunday. Each generator has its own vertical scale because its traffic volume differs; compare changes within each trace before comparing absolute heights.

The lower row shows each generator's daily totals on its own scale. The static generator samples the same rules throughout the week. The workload generator explicitly reduces weekend demand, although random bursts and batches affect individual days.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for column, (name, hours) in enumerate(weekly_hours_by_generator.items()):
    ax = axes[0, column]
    ax.plot(hours.index + 0.5, hours["CSV rows"] / 3600, linewidth=1.2)
    ax.axvspan(120, 168, color="grey", alpha=0.15, label="Weekend")
    for boundary in range(0, 169, 24):
        ax.axvline(boundary, color="grey", alpha=0.2, linewidth=0.7)
    ax.set_xticks(np.arange(7) * 24 + 12, DAY_NAMES)
    ax.set_xlim(0, 168)
    ax.set_ylim(bottom=0)
    ax.set_title(name + ": full week")
    ax.set_ylabel("CSV rows / simulated second (hourly mean)")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    daily = weekly_daily_rows[name]
    axes[1, column].bar(DAY_NAMES, daily, color=["tab:blue"] * 5 + ["tab:orange"] * 2)
    axes[1, column].set_ylabel("CSV rows / complete day")
    axes[1, column].set_title(name + ": daily totals")
    axes[1, column].grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 12. Inspect each day: seven rows, two generators

Each row below represents the **same day** in the two traces. The left column is the static generator; the right column is the workload generator. The horizontal axis is the hour within that day.

Vertical scales are shared down each column: Monday and Sunday from the same generator use identical scales. The two columns use different scales because the generators have different baseline volumes. Look for daily peaks and the weekend change in the right column, and the absence of an explicit weekday/weekend pattern in the left column.

In [ ]:
fig, axes = plt.subplots(7, 2, figsize=(13, 16), sharex=True, sharey="col")
hours_of_day = np.arange(24) + 0.5
for column, (name, hours) in enumerate(weekly_hours_by_generator.items()):
    day_hour_rates = hours["CSV rows"].to_numpy().reshape(7, 24) / 3600
    for day, day_name in enumerate(DAY_NAMES):
        ax = axes[day, column]
        color = "tab:orange" if day >= 5 else "tab:blue"
        ax.plot(hours_of_day, day_hour_rates[day], color=color, linewidth=1.2)
        ax.set_title(f"{day_name} — {name}", fontsize=10)
        ax.set_ylabel("CSV rows / second")
        ax.set_xlim(0, 24)
        ax.set_ylim(bottom=0)
        ax.set_xticks(range(0, 25, 4))
        ax.grid(alpha=0.3)
        if day >= 5:
            ax.set_facecolor("#fff6e8")
        if day == 6:
            ax.set_xlabel("Simulated hour of day")
fig.tight_layout()
plt.show()

## 13. Compare relative shapes and quantify the weekend change

To compare **shape** despite different traffic volumes, divide each generator's hourly rate by its own average weekday rate. A value of `1` is that generator's weekday baseline, and `0.5` is half of it. This normalization is applied only to this figure; the preceding figures show raw observed counts and rates.

For the numerical weekend comparison, average the five complete weekdays and the two complete weekend days separately. Comparing the two sums would confound traffic intensity with the different number of days.

`Workload.shape()` applies weekend multipliers to the daily humps of IW-F (`0.08`) and IW-N (`0.10`), retaining their baseline traffic. NIW background demand uses a `0.75` weekend multiplier; additional batch-drop probabilities are not weekend-reduced. Therefore the model lowers expected weekend traffic, but these numbers are not exact ratios for total sampled request counts. The static generator has no calendar-dependent rule. Randomness still makes its daily counts differ slightly.

In [ ]:
week_comparison_rows = []
fig, ax = plt.subplots(figsize=(13, 4))
for name, hours in weekly_hours_by_generator.items():
    daily = weekly_daily_rows[name]
    weekday_daily_mean = daily.iloc[:5].mean()
    weekend_daily_mean = daily.iloc[5:].mean()
    ratio = weekend_daily_mean / weekday_daily_mean if weekday_daily_mean else np.nan
    week_comparison_rows.append({
        "Generator": name,
        "Weekday mean CSV rows/day": weekday_daily_mean,
        "Weekend mean CSV rows/day": weekend_daily_mean,
        "Weekend / weekday": ratio,
        "Weekend reduction (%)": (1 - ratio) * 100,
    })
    hourly_rate = hours["CSV rows"] / 3600
    baseline_rate = hourly_rate.iloc[:120].mean()
    if baseline_rate > 0:
        ax.plot(hours.index + 0.5, hourly_rate / baseline_rate, label=name, linewidth=1.2)

ax.axvspan(120, 168, color="grey", alpha=0.15, label="Weekend")
ax.axhline(1, color="black", linestyle="--", linewidth=0.8, label="Own weekday mean")
ax.set_xticks(np.arange(7) * 24 + 12, DAY_NAMES)
ax.set_xlim(0, 168)
ax.set_ylim(bottom=0)
ax.set_ylabel("Hourly arrival rate / own weekday mean rate")
ax.set_title("Relative weekly patterns from both saved traces")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

weekly_comparison = pd.DataFrame(week_comparison_rows).set_index("Generator")
display(weekly_comparison.round(3))

The weekly comparison is now based on **two generated CSV traces**, analyzed with the same bucketing code. Change `WEEK_SEED` and rerun sections 9–13 to inspect another realization. A single sample illustrates the behavior; the generator implementation explains which patterns are intentional.

The full files remain accessible through `weekly_trace_paths` while this kernel keeps `weekly_csv_storage` alive. For example, use `pd.read_csv(weekly_trace_paths["Static generator"], nrows=10, dtype={"regions": "string"})` to inspect its first ten rows. Your existing versioned trace is untouched.